# Continuing Rejection Sampling with MCMC

Rejection sampling is fast and parallelisable, but its efficiency drops when the
posterior is narrow relative to the prior -- the acceptance rate falls and you need
to draw many more prior samples to collect enough posterior samples.

A practical workflow is therefore:

1. **Rejection sampling** -- fast exploration; identifies the posterior mode and
   gives a rough characterisation of the posterior shape.
2. **MCMC** -- warm-started from the rejection-sampler output; efficient sampling
   of the exact posterior without any accept/reject waste.

`harv` supports this workflow through `NumpyroSampler.init_mcmc`, which
automatically builds a numpyro model from the existing prior and data, seeds
each MCMC chain from a rejection-sampler posterior draw, and returns a thin
`WarmStartMCMC` wrapper that injects those starting positions into
`numpyro.infer.MCMC.run()`.

Two model variants are available:

| `marginalized` | What MCMC samples | Sample sites |
|---|---|---|
| `True` (default) | Nonlinear parameters only; linear params analytically marginalized | `period`, `eccentricity`, ... |
| `False` | All parameters jointly | above + `K`, `v0` / astrometric solution |

This notebook demonstrates both modes for RV and astrometry data.

In [ ]:
import jax
import jax.random as jr
import matplotlib.pyplot as plt
from unxt import Quantity

jax.config.update("jax_enable_x64", True)

from harv import Model
from harv.priors import RejectionPrior
from harv.samplers import NumpyroSampler, RejectionSampler
from harv.simulate import simulate_gaia_epoch_astrometry, simulate_rv_sb1_data

---
## Part 1: Radial velocity

In [ ]:
# Simulate RV observations for an SB1 system.
rv_data, rv_true = simulate_rv_sb1_data(
    seed=0,
    n_obs=30,
    period=Quantity(180.0, "day"),
    eccentricity=0.3,
    K=Quantity(12.0, "km/s"),
    v0=Quantity(-5.0, "km/s"),
    rv_error=Quantity(0.5, "km/s"),
)
print(f"RV observations: {len(rv_data.time)}")
print(f"True period: {rv_true['period']}")

In [ ]:
# Set up the prior, model, and run rejection sampling.
rv_prior = RejectionPrior.default_rv(period_min=50.0, period_max=500.0)
rv_model = Model(rv_prior, rv_data)
rv_sampler = RejectionSampler(rv_model)

rv_samples = rv_sampler.run(n_prior_samples=500_000)
print(f"Accepted samples: {rv_samples.n_samples}")
print(f"Acceptance rate:  {rv_samples.n_samples / 500_000:.4%}")

### 1a. RV — marginalized MCMC (default)

MCMC explores only the nonlinear subspace (`period`, `eccentricity`, `phase_peri`,
`arg_peri`). Linear parameters (`K`, `v0`) remain analytically marginalized,
exactly as in rejection sampling. This is the lowest-dimensional option and
usually the fastest to mix.

In [ ]:
rv_mcmc_sampler = NumpyroSampler(rv_model)
mcmc_rv_marg = rv_mcmc_sampler.init_mcmc(
    rv_samples,
    # marginalized=True is the default -- shown explicitly here for clarity.
    marginalized=True,
    num_chains=4,
    num_warmup=500,
    num_samples=1_000,
    chain_method="sequential",  # use "parallel" if multiple devices are available
)

# run() automatically injects the rejection-sampler positions as init_params.
mcmc_rv_marg.run(jr.key(1))
mcmc_rv_marg.print_summary()

In [ ]:
posterior_rv_marg = mcmc_rv_marg.get_samples()
print("Sites in posterior:", list(posterior_rv_marg.keys()))
print("period shape:", posterior_rv_marg["period"].shape)  # (num_chains * num_samples,)

### 1b. RV — full (unmarginalized) MCMC

MCMC samples all parameters jointly. Linear parameters are drawn from a joint
`MultivariateNormal` latent site `"_linear"` and exposed as named deterministic
sites (`"K"`, `"v0"`). This higher-dimensional space mixes more slowly but
produces explicit posterior samples for the linear parameters.

In [ ]:
mcmc_rv_full = rv_mcmc_sampler.init_mcmc(
    rv_samples,
    marginalized=False,
    num_chains=4,
    num_warmup=500,
    num_samples=1_000,
    chain_method="sequential",
)

mcmc_rv_full.run(jr.key(1))
mcmc_rv_full.print_summary()

In [ ]:
posterior_rv_full = mcmc_rv_full.get_samples()
print("Sites in posterior:", list(posterior_rv_full.keys()))
# K and v0 appear as deterministic sites; "_linear" is the joint latent.
print("K  (km/s) — mean:", posterior_rv_full["K"].mean())
print("v0 (km/s) — mean:", posterior_rv_full["v0"].mean())

In [ ]:
# Compare period posteriors between the two modes — they should agree.
fig, ax = plt.subplots(figsize=(7, 3))
ax.hist(posterior_rv_marg["period"], bins=40, alpha=0.6, label="marginalized")
ax.hist(posterior_rv_full["period"], bins=40, alpha=0.6, label="full")
ax.axvline(float(rv_true["period"].to_value("day")), color="k", ls="--", label="truth")
ax.set_xlabel("period (days)")
ax.legend()
fig.tight_layout()

---
## Part 2: Gaia astrometry

In [ ]:
# Simulate Gaia-like epoch astrometry.
astro_data, astro_true = simulate_gaia_epoch_astrometry(
    seed=7,
    n_obs=50,
    period=Quantity(365.25, "day"),
    eccentricity=0.2,
    inclination=Quantity(50.0, "deg"),
    arg_peri=Quantity(30.0, "deg"),
    lon_asc_node=Quantity(90.0, "deg"),
    semimajor_axis=Quantity(2.0, "mas"),
    parallax=Quantity(8.0, "mas"),
    mu_alpha=Quantity(3.0, "mas/yr"),
    mu_delta=Quantity(-2.0, "mas/yr"),
    al_error=Quantity(0.3, "mas"),
)
print(f"Astrometry observations: {len(astro_data.time)}")
print(f"True period: {astro_true['period']}")

In [ ]:
# Set up prior, model, and run rejection sampling.
astro_prior = RejectionPrior.default_astrometry(period_min=100.0, period_max=1000.0)
astro_model = Model(astro_prior, astro_data)
astro_sampler = RejectionSampler(astro_model)

astro_samples = astro_sampler.run(n_prior_samples=500_000)
print(f"Accepted samples: {astro_samples.n_samples}")
print(f"Acceptance rate:  {astro_samples.n_samples / 500_000:.4%}")

### 2a. Astrometry — marginalized MCMC (default)

MCMC explores the six nonlinear parameters (`period`, `eccentricity`, `phase_peri`,
`arg_peri`, `cos_i`, `lon_asc_node`). The six linear parameters (RA offset, Dec
offset, proper motions, parallax, semi-major axis) are analytically marginalized.

In [ ]:
astro_mcmc_sampler = NumpyroSampler(astro_model)
mcmc_astro_marg = astro_mcmc_sampler.init_mcmc(
    astro_samples,
    marginalized=True,
    num_chains=4,
    num_warmup=500,
    num_samples=1_000,
    chain_method="sequential",
)

mcmc_astro_marg.run(jr.key(2))
mcmc_astro_marg.print_summary()

In [ ]:
posterior_astro_marg = mcmc_astro_marg.get_samples()
print("Sites in posterior:", list(posterior_astro_marg.keys()))

### 2b. Astrometry — full (unmarginalized) MCMC

MCMC samples all 12 parameters jointly. The six astrometric linear parameters
(`ra0`, `dec0`, `pmra`, `pmdec`, `parallax`, `semi_major_axis`) appear as named
deterministic sites in `get_samples()`.

In [ ]:
mcmc_astro_full = astro_mcmc_sampler.init_mcmc(
    astro_samples,
    marginalized=False,
    num_chains=4,
    num_warmup=500,
    num_samples=1_000,
    chain_method="sequential",
)

mcmc_astro_full.run(jr.key(2))
mcmc_astro_full.print_summary()

In [ ]:
posterior_astro_full = mcmc_astro_full.get_samples()
print("Sites in posterior:", list(posterior_astro_full.keys()))
# Named linear deterministic sites:
print("parallax (mas) — mean:", posterior_astro_full["parallax"].mean())
print("semi_major_axis (mas) — mean:", posterior_astro_full["semi_major_axis"].mean())

In [ ]:
# Compare period posteriors between the two modes.
fig, ax = plt.subplots(figsize=(7, 3))
ax.hist(posterior_astro_marg["period"], bins=40, alpha=0.6, label="marginalized")
ax.hist(posterior_astro_full["period"], bins=40, alpha=0.6, label="full")
ax.axvline(365.25, color="k", ls="--", label="truth")
ax.set_xlabel("period (days)")
ax.legend()
fig.tight_layout()

---
## Summary

| Scenario | Code |
|---|---|
| RV, marginalized | `NumpyroSampler(model).init_mcmc(samples)` |
| RV, full | `NumpyroSampler(model).init_mcmc(samples, marginalized=False)` |
| Astrometry, marginalized | `NumpyroSampler(model).init_mcmc(samples)` |
| Astrometry, full | `NumpyroSampler(model).init_mcmc(samples, marginalized=False)` |

The same pattern works for combined astrometry + RV data (`SourceData`).

**Which mode to choose?**

- Use `marginalized=True` when you only care about the nonlinear orbital parameters
  (period, eccentricity, inclination, ...). It is lower-dimensional and mixes faster.
- Use `marginalized=False` when you need explicit posterior samples for the linear
  parameters (K, v0, astrometric solution), for example to propagate uncertainties
  into derived quantities or to compare with independent estimates.